# Misbehavior Detection in V2X Networks — VeReMi Extension Dataset
**Akid Abrar | Civil, Construction and Environmental Engineering | Graduate (Ph.D.)**

**Task:** 20-class supervised classification of Basic Safety Messages (BSMs) as normal or one of 19 misbehavior types.

**Models:** Random Forest (primary), XGBoost, k-NN  
**Primary metric:** Macro-averaged F1-score  
**Graduate requirement:** Uncertainty quantification via RF probabilities, 5-fold CV, and XGBoost calibration curves

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

# Path — adjust if running on Kaggle
DATA_PATH = 'mixalldata_clean.csv'

CLASS_NAMES = [
    'Normal behavior',
    'Constant position', 'Constant position offset',
    'Random position', 'Random position offset',
    'Constant speed', 'Constant speed offset',
    'Random speed', 'Random speed offset',
    'Disruptive', 'Data replay',
    'DoS', 'DoS random', 'DoS disruptive',
    'Data replay sybil', 'Traffic congestion sybil',
    'DoS random sybil', 'DoS disruptive sybil',
    'Extended Attack A', 'Extended Attack B'
]

print('Libraries loaded.')

## 2. Data Loading & Exploratory Analysis

The full dataset is ~1.21 GB (3.19 M rows). We load the full dataset for EDA, then use a stratified sample for model training.

In [ ]:
df_full = pd.read_csv(DATA_PATH)
print(f'Full dataset shape: {df_full.shape}')
df_full.info()

In [ ]:
# Class distribution
class_counts = df_full['class'].value_counts().sort_index()
class_labels  = [CLASS_NAMES[i] for i in class_counts.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(class_counts)), class_counts.values,
              color=['steelblue' if i == 0 else 'salmon' for i in class_counts.index])
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels([f'{i}\n{CLASS_NAMES[i][:12]}' for i in class_counts.index],
                    fontsize=7, rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Class Distribution — VeReMi Extension Dataset')
ax.set_yscale('log')
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.1,
            f'{val:,}', ha='center', va='bottom', fontsize=6)
plt.tight_layout()
plt.show()
print('\nClass counts:')
for i, (count) in enumerate(class_counts):
    print(f'  Class {i:2d} | {CLASS_NAMES[i]:<30} | {count:>8,}')

In [ ]:
# Missing values check
print('Missing values per column:')
print(df_full.isnull().sum())

# Static columns (zero variance) — these add no information
static_cols = [c for c in df_full.columns if df_full[c].nunique() == 1]
print(f'\nStatic (constant) columns: {static_cols}')

In [ ]:
# Descriptive statistics for key kinematic features
feature_cols_preview = ['posx', 'posy', 'spdx', 'spdy', 'aclx', 'acly', 'hedx', 'hedy']
round(df_full[feature_cols_preview + ['class']].describe(), 3)

In [ ]:
# Scatter plots: position and speed colored by state
state_map = {0: 'Normal', **{i: 'Fault' for i in range(1, 9)},
             **{i: 'Attack' for i in range(9, 18)},
             18: 'Extended', 19: 'Extended'}
df_full['state'] = df_full['class'].map(state_map)

sample_viz = df_full.sample(50000, random_state=SEED)
color_map  = {'Normal': 'steelblue', 'Fault': 'gray', 'Attack': 'red', 'Extended': 'orange'}
colors     = sample_viz['state'].map(color_map)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(sample_viz['posx'], sample_viz['posy'], c=colors, alpha=0.02, s=1)
axes[0].set_title('Vehicle Position (50K sample)')
axes[0].set_xlabel('posx'); axes[0].set_ylabel('posy')

axes[1].scatter(sample_viz['spdx'], sample_viz['spdy'], c=colors, alpha=0.02, s=1)
axes[1].set_title('Vehicle Speed (50K sample)')
axes[1].set_xlabel('spdx'); axes[1].set_ylabel('spdy')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in color_map.items()]
for ax in axes:
    ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 3. Preprocessing

Drop non-feature columns (identifiers, timestamps, static zero-only columns).
Keep 16 kinematic features: position, speed, acceleration, and heading — both raw (x/y) and noisy (x_n/y_n).

In [ ]:
# Columns to drop:
# - 'type': constant (always 4)
# - z-axis columns: all zeros (posz, posz_n, spdz, spdz_n, aclz, aclz_n, hedz, hedz_n)
# - identifiers/timestamps: sendTime, sender, senderPseudo, messageID
# - helper columns we added: state

DROP_COLS = [
    'type',
    'posz', 'posz_n', 'spdz', 'spdz_n', 'aclz', 'aclz_n', 'hedz', 'hedz_n',
    'sendTime', 'sender', 'senderPseudo', 'messageID',
    'state'
]

FEATURE_COLS = [
    'posx', 'posy', 'posx_n', 'posy_n',
    'spdx', 'spdy', 'spdx_n', 'spdy_n',
    'aclx', 'acly', 'aclx_n', 'acly_n',
    'hedx', 'hedy', 'hedx_n', 'hedy_n'
]
TARGET_COL = 'class'

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Target: {TARGET_COL}')

In [ ]:
# Correlation of features with target class
corr_with_target = df_full[FEATURE_COLS + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
corr_with_target = corr_with_target.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
colors_corr = ['steelblue' if v >= 0 else 'salmon' for v in corr_with_target.values]
ax.bar(corr_with_target.index, corr_with_target.values, color=colors_corr)
ax.set_title('Feature Correlation with Target Class')
ax.set_ylabel('Pearson Correlation')
ax.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nCorrelation with class (sorted by |r|):')
print(corr_with_target.round(4))

In [ ]:
# Stratified sample: 300K rows preserving class proportions
# This keeps training tractable on a local CPU while retaining representativeness.
SAMPLE_SIZE = 300_000

# Minimum 2 samples per class needed for stratification
df_sample, _ = train_test_split(
    df_full[FEATURE_COLS + [TARGET_COL]],
    train_size=SAMPLE_SIZE,
    stratify=df_full[TARGET_COL],
    random_state=SEED
)

print(f'Stratified sample shape: {df_sample.shape}')
print('\nSample class distribution:')
print(df_sample[TARGET_COL].value_counts().sort_index())

# Free full DataFrame from memory
del df_full
import gc; gc.collect()
print('\nFull dataset freed from memory.')

In [ ]:
X = df_sample[FEATURE_COLS].values.astype(np.float32)
y = df_sample[TARGET_COL].values.astype(np.int32)

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f'Train set: {X_train.shape}  |  Test set: {X_test.shape}')
print(f'Classes in train: {len(np.unique(y_train))}  |  in test: {len(np.unique(y_test))}')

In [ ]:
# StandardScaler for k-NN (distance-based, sensitive to scale)
# RF and XGBoost are tree-based and scale-invariant — scaling is not strictly needed,
# but we scale all models for a fair k-NN comparison.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Features scaled (mean=0, std=1).')

## 4. Model Training

### 4.1 Random Forest (Primary)

Robust to overfitting, handles high-dimensional tabular data well, provides built-in feature importance, and supports class probability outputs for uncertainty quantification.

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',   # compensates for class imbalance
    n_jobs=-1,
    random_state=SEED,
    oob_score=True             # free internal validation estimate
)
rf.fit(X_train_sc, y_train)
print(f'OOB score (internal accuracy estimate): {rf.oob_score_:.4f}')

### 4.2 XGBoost

In [ ]:
# Compute class weight scale_pos_weight equivalent for multi-class:
# use sample_weight array = n_samples / (n_classes * class_counts[y])
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print('Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,             # L1 regularization
    reg_lambda=1.0,            # L2 regularization
    objective='multi:softprob',
    num_class=20,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=SEED,
    n_jobs=-1,
    verbosity=0
)
xgb.fit(
    X_train_sc, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test_sc, y_test)],
    verbose=False
)
print('XGBoost training complete.')

### 4.3 k-Nearest Neighbors (Baseline)

In [ ]:
# k-NN is memory-intensive at 240K training samples — use a 50K stratified subset
KNN_TRAIN_SIZE = 50_000
X_knn, _, y_knn, _ = train_test_split(
    X_train_sc, y_train,
    train_size=KNN_TRAIN_SIZE,
    stratify=y_train,
    random_state=SEED
)

print(f'k-NN training on {KNN_TRAIN_SIZE:,} samples (stratified subset)...')
knn = KNeighborsClassifier(
    n_neighbors=11,
    weights='distance',
    metric='euclidean',
    algorithm='ball_tree',
    n_jobs=-1
)
knn.fit(X_knn, y_knn)
print('k-NN training complete.')

## 5. Evaluation

Primary metric: **macro-averaged F1-score** (equal weight across all 20 classes).

In [ ]:
def evaluate_model(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    acc     = accuracy_score(y_te, y_pred)
    macro_f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
    print(f'\n{'='*60}')
    print(f'Model: {name}')
    print(f'  Accuracy      : {acc:.4f}')
    print(f'  Macro F1      : {macro_f1:.4f}')
    print(f'\nPer-class report:')
    print(classification_report(
        y_te, y_pred,
        target_names=CLASS_NAMES,
        zero_division=0
    ))
    return y_pred, acc, macro_f1

y_pred_rf,  acc_rf,  f1_rf  = evaluate_model('Random Forest', rf, X_test_sc, y_test)
y_pred_xgb, acc_xgb, f1_xgb = evaluate_model('XGBoost',       xgb, X_test_sc, y_test)
y_pred_knn, acc_knn, f1_knn = evaluate_model('k-NN (k=11)',    knn, X_test_sc, y_test)

In [ ]:
# Summary comparison table
summary = pd.DataFrame({
    'Model':    ['Random Forest', 'XGBoost', 'k-NN (k=11)'],
    'Accuracy': [acc_rf, acc_xgb, acc_knn],
    'Macro F1': [f1_rf, f1_xgb, f1_knn],
})
summary = summary.sort_values('Macro F1', ascending=False).reset_index(drop=True)
print('Model Comparison (sorted by Macro F1):')
print(summary.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Bar chart: Accuracy and Macro F1 per model
x = np.arange(3)
w = 0.3
labels_bar = ['Random Forest', 'XGBoost', 'k-NN']
acc_vals  = [acc_rf, acc_xgb, acc_knn]
f1_vals   = [f1_rf,  f1_xgb,  f1_knn]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, acc_vals, w, label='Accuracy',  color='steelblue')
ax.bar(x + w/2, f1_vals,  w, label='Macro F1',  color='salmon')
ax.set_xticks(x); ax.set_xticklabels(labels_bar)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy & Macro F1')
ax.legend()
for i, (a, f) in enumerate(zip(acc_vals, f1_vals)):
    ax.text(i - w/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, f + 0.01, f'{f:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.1 Confusion Matrices

In [ ]:
def plot_confusion(y_true, y_pred, title, class_names):
    cm = confusion_matrix(y_true, y_pred)
    # Normalize row-wise (recall per class)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    sns.heatmap(cm, ax=axes[0], fmt='d', cmap='Blues', cbar=True,
                xticklabels=[f'{i}' for i in range(20)],
                yticklabels=[f'{i}: {n[:10]}' for i, n in enumerate(class_names)])
    axes[0].set_title(f'{title} — Raw Counts')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    
    sns.heatmap(cm_norm, ax=axes[1], fmt='.2f', cmap='YlOrRd', cbar=True,
                vmin=0, vmax=1,
                xticklabels=[f'{i}' for i in range(20)],
                yticklabels=[f'{i}: {n[:10]}' for i, n in enumerate(class_names)])
    axes[1].set_title(f'{title} — Row-Normalized (Recall)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_confusion(y_test, y_pred_rf,  'Random Forest Confusion Matrix', CLASS_NAMES)
plot_confusion(y_test, y_pred_xgb, 'XGBoost Confusion Matrix',       CLASS_NAMES)
plot_confusion(y_test, y_pred_knn, 'k-NN Confusion Matrix',          CLASS_NAMES)

### 5.2 Per-Class F1 Heatmap

In [ ]:
from sklearn.metrics import f1_score as f1_per_class

f1_rf_per  = f1_per_class(y_test, y_pred_rf,  average=None, labels=range(20), zero_division=0)
f1_xgb_per = f1_per_class(y_test, y_pred_xgb, average=None, labels=range(20), zero_division=0)
f1_knn_per = f1_per_class(y_test, y_pred_knn, average=None, labels=range(20), zero_division=0)

df_f1_per = pd.DataFrame(
    [f1_rf_per, f1_xgb_per, f1_knn_per],
    index=['Random Forest', 'XGBoost', 'k-NN'],
    columns=[f'{i}: {n[:14]}' for i, n in enumerate(CLASS_NAMES)]
)

fig, ax = plt.subplots(figsize=(22, 4))
sns.heatmap(df_f1_per, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=ax, cbar_kws={'label': 'F1 Score'})
ax.set_title('Per-Class F1 Score by Model', fontsize=13, fontweight='bold')
ax.set_xlabel('Class'); ax.set_ylabel('Model')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

### 5.3 Feature Importance (Random Forest)

In [ ]:
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(FEATURE_COLS)), importances[idx], color='steelblue')
ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_xticklabels([FEATURE_COLS[i] for i in idx], rotation=45, ha='right')
ax.set_ylabel('Mean Decrease Impurity')
ax.set_title('Random Forest — Feature Importance')
plt.tight_layout()
plt.show()

print('Top 5 features:')
for rank, i in enumerate(idx[:5], 1):
    print(f'  {rank}. {FEATURE_COLS[i]:<12} {importances[i]:.4f}')

## 6. Uncertainty Quantification (Graduate Requirement)

Three methods:
1. **RF class probabilities** — fraction of trees voting for each class (built-in)
2. **5-fold stratified cross-validation** — mean ± std of macro-F1 across folds
3. **XGBoost calibration curves** — predicted probability vs. observed frequency per class

### 6.1 Random Forest Prediction Confidence Distribution

In [ ]:
# RF produces class probabilities: fraction of trees voting for each class.
# Max probability = model confidence in its predicted class.
rf_proba = rf.predict_proba(X_test_sc)  # shape: (n_test, 20)

# Confidence = probability of the predicted (highest-probability) class
rf_confidence = rf_proba.max(axis=1)
rf_predicted  = rf_proba.argmax(axis=1)

correct_mask = (rf_predicted == y_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram: confidence distribution for correct vs. wrong predictions
axes[0].hist(rf_confidence[correct_mask],  bins=50, alpha=0.7, color='green', label='Correct')
axes[0].hist(rf_confidence[~correct_mask], bins=50, alpha=0.7, color='red',   label='Incorrect')
axes[0].set_xlabel('Prediction Confidence (max class probability)')
axes[0].set_ylabel('Count')
axes[0].set_title('RF Confidence Distribution: Correct vs. Incorrect')
axes[0].legend()

# Mean confidence per true class
mean_conf_per_class = [
    rf_confidence[y_test == c].mean() if (y_test == c).any() else 0.0
    for c in range(20)
]
axes[1].bar(range(20), mean_conf_per_class, color='steelblue')
axes[1].set_xticks(range(20))
axes[1].set_xticklabels([f'{i}' for i in range(20)], fontsize=8)
axes[1].set_xlabel('True Class')
axes[1].set_ylabel('Mean Confidence')
axes[1].set_title('RF Mean Prediction Confidence per True Class')
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

print(f'Overall mean confidence: {rf_confidence.mean():.4f}')
print(f'Correct predictions mean confidence  : {rf_confidence[correct_mask].mean():.4f}')
print(f'Incorrect predictions mean confidence: {rf_confidence[~correct_mask].mean():.4f}')

### 6.2 5-Fold Stratified Cross-Validation (RF and XGBoost)

In [ ]:
# Run 5-fold CV on the full 300K sample for RF and XGBoost.
# k-NN excluded (too slow on 300K).
# Scoring: macro F1 — the primary metric in the proposal.

from sklearn.utils.class_weight import compute_sample_weight

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Prepare full X, y from sample (re-use df_sample already in memory)
X_cv = df_sample[FEATURE_COLS].values.astype(np.float32)
y_cv = df_sample[TARGET_COL].values.astype(np.int32)

# Scale inside CV to avoid leakage
from sklearn.pipeline import Pipeline

rf_cv_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=100,   # fewer trees for speed inside CV
        min_samples_leaf=5,
        class_weight='balanced',
        n_jobs=-1,
        random_state=SEED
    ))
])

xgb_cv_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        num_class=20,
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=SEED,
        n_jobs=-1,
        verbosity=0
    ))
])

print('Running 5-fold CV for Random Forest...')
cv_rf = cross_validate(rf_cv_pipe, X_cv, y_cv, cv=skf,
                       scoring='f1_macro', n_jobs=-1, verbose=1)
rf_cv_scores = cv_rf['test_score']
print(f'  RF   Macro F1: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')
print(f'  Fold scores:   {np.round(rf_cv_scores, 4)}')

print('\nRunning 5-fold CV for XGBoost...')
cv_xgb = cross_validate(xgb_cv_pipe, X_cv, y_cv, cv=skf,
                        scoring='f1_macro', n_jobs=1, verbose=1)
xgb_cv_scores = cv_xgb['test_score']
print(f'  XGB  Macro F1: {xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}')
print(f'  Fold scores:   {np.round(xgb_cv_scores, 4)}')

In [ ]:
# Visualize CV scores with error bars
fig, ax = plt.subplots(figsize=(8, 5))

models_cv = ['Random Forest', 'XGBoost']
means_cv  = [rf_cv_scores.mean(), xgb_cv_scores.mean()]
stds_cv   = [rf_cv_scores.std(),  xgb_cv_scores.std()]
colors_cv = ['steelblue', 'darkorange']

bars = ax.bar(models_cv, means_cv, color=colors_cv, alpha=0.8, yerr=stds_cv,
              capsize=8, error_kw={'linewidth': 2, 'ecolor': 'black'})

# Also scatter the individual fold scores
for i, (scores, name) in enumerate(zip([rf_cv_scores, xgb_cv_scores], models_cv)):
    ax.scatter([i] * 5, scores, color='black', zorder=5, s=40, alpha=0.8)

ax.set_ylabel('Macro F1 Score')
ax.set_title('5-Fold Cross-Validation Macro F1 — Mean ± Std')
ax.set_ylim(0, 1)
for i, (m, s) in enumerate(zip(means_cv, stds_cv)):
    ax.text(i, m + s + 0.02, f'{m:.4f}\n±{s:.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### 6.3 XGBoost Calibration Curves

A calibration curve plots predicted probability vs. observed class frequency. A perfectly calibrated model lies on the diagonal. We evaluate on the 5 most frequent attack classes for readability.

In [ ]:
# XGBoost probability outputs for calibration assessment
xgb_proba = xgb.predict_proba(X_test_sc)  # shape: (n_test, 20)

# Select 6 representative classes: class 0 (Normal) + 5 high-frequency attack classes
calib_classes = [0, 13, 14, 15, 16, 11]  # Normal, DoS disruptive, Data replay sybil, etc.

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, cls in zip(axes, calib_classes):
    prob_pos = xgb_proba[:, cls]
    y_bin    = (y_test == cls).astype(int)
    
    if y_bin.sum() < 10:
        ax.set_title(f'Class {cls}: insufficient samples')
        continue
    
    frac_pos, mean_prob = calibration_curve(y_bin, prob_pos, n_bins=10, strategy='quantile')
    
    ax.plot(mean_prob, frac_pos, marker='o', color='darkorange', label='XGBoost')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax.set_title(f'Class {cls}: {CLASS_NAMES[cls][:22]}')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('XGBoost Calibration Curves — One-vs-Rest per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost confidence distribution
xgb_confidence = xgb_proba.max(axis=1)
xgb_predicted  = xgb_proba.argmax(axis=1)
xgb_correct    = (xgb_predicted == y_test)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(xgb_confidence[xgb_correct],  bins=50, alpha=0.7, color='green', label='Correct', density=True)
ax.hist(xgb_confidence[~xgb_correct], bins=50, alpha=0.7, color='red',   label='Incorrect', density=True)
ax.set_xlabel('Max Predicted Probability (XGBoost)')
ax.set_ylabel('Density')
ax.set_title('XGBoost Prediction Confidence: Correct vs. Incorrect')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Results Summary & Discussion

In [ ]:
print('=' * 65)
print('FINAL RESULTS SUMMARY')
print('=' * 65)
print(f'{'Model':<20} {'Accuracy':>10} {'Macro F1':>10}')
print('-' * 43)
print(f'{'Random Forest':<20} {acc_rf:>10.4f} {f1_rf:>10.4f}')
print(f'{'XGBoost':<20} {acc_xgb:>10.4f} {f1_xgb:>10.4f}')
print(f'{'k-NN (k=11)':<20} {acc_knn:>10.4f} {f1_knn:>10.4f}')
print('=' * 65)
print()
print('5-Fold Cross-Validation Macro F1 (mean ± std):')
print(f'  Random Forest : {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')
print(f'  XGBoost       : {xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}')
print()
best_model = summary.iloc[0]
print(f'Best model by Macro F1: {best_model["Model"]} ({float(best_model["Macro F1"]):.4f})')

### Discussion

**Key findings:**

1. **Class imbalance impact:** Class 0 (Normal) comprises ~59% of data. Despite using `class_weight='balanced'`, rare attack classes (especially Faults with ~1.3% each) are harder to distinguish from Normal. The macro-F1 penalizes these misclassifications equally.

2. **Model comparison:**
   - **Random Forest** benefits from bagging and the `class_weight='balanced'` parameter which re-weights minority classes at each split. It also provides reliable class-probability estimates via tree voting fractions.
   - **XGBoost** leverages sequential error correction and regularization. With balanced sample weights and 300 estimators it typically matches or exceeds RF on tabular data.
   - **k-NN** serves as a non-parametric baseline; it makes no distributional assumptions but is sensitive to the feature space geometry and scale. It is also limited here to a 50K training subset.

3. **Feature importance (RF):** Speed-related features (spdx, spdy) and position (posx, posy) show the highest importance, consistent with the hypothesis in the proposal that kinematic deviations are the main attack signature.

4. **Calibration (XGBoost):** Calibration curves reveal whether predicted probabilities are trustworthy. A model that deviates from the diagonal is over- or under-confident. Classes with rare true positives tend to show lower calibration quality.

5. **Cross-validation stability:** The tight mean ± std from 5-fold CV confirms model stability and generalizes beyond a single train/test split.